In [ ]:
# 오래된 이미지 -> BSRGAN -> colorization -> (과정시각화?) -> GPT(혹은 챗봇)을 통해 자연어 생성

In [ ]:
# https://huggingface.co/Hammad712/GAN-Colorization-Model
from huggingface_hub import from_pretrained_fastai

learn = from_pretrained_fastai("Hammad712/GAN-Colorization-Model")

In [ ]:
http://210.125.70.71:30880/login
estsoft13~24
!Passw0rd

c:\Users\devchoi\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\devchoi\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\devchoi\.cache\huggingface\hub\models--davidramos--image-colorization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For 

In [ ]:
https://www.learnmore.co.kr/education/ai-literacy/machine-learning/reinforcement-learning/deoldify
https://github.com/jantic/DeOldify



In [ ]:
# 13 김민섭
# 14 양혁준
# 15 김현우
# 16 김주형
# 17 손동빈
# 18 이동현

# 온라인
# 20 길장호
# 24 강예진


In [ ]:
# 사진 복원 + 스토리 생성 프로젝트 전체 정리 (찐찐찐찐찐찐찐찐찐찐찐찐막 수정)
# 1. 프로젝트 개요
# * 사용자 입력: 오래된 사진 (흑백 or 컬러) + 간략 설명(한국어)
# * 목표:
#     1. 사진 복원 (흑백→컬러, 노이즈 제거, 해상도 업)
#     2. 복원된 이미지를 기반으로 스토리 생성 (최종 한국어 출력)

# 2. 프로세스 플로우
# (A) 흑백 사진
# 1. 필수: 컬러 복원 (DeOldify 등)
# 2. 선택 버튼:
#     * 해상도 업 (최대 3회)
#     * 노이즈 제거 (최대 3회)
#     * 스토리 생성
# (B) 컬러 사진
# 1. 필수 없음 (원본 그대로 시작)
# 2. 선택 버튼:
#     * 해상도 업 (최대 3회)
#     * 노이즈 제거 (최대 3회)
#     * 스토리 생성
# 공통 규칙
# * 각 결과 이미지 아래 현재 상태 표시
# [컬러화 ✔ / 해상도 1회 / 노이즈 0회]

# * 최종 출력: 복원 이미지 + 스토리

# 3. 사용 기술 후보
# * 복원 모델:
#     * DeOldify (흑백 → 컬러 복원)
#     * ESRGAN (해상도 업)
#     * Palette / NAFNet (노이즈 제거)
# * 스토리 생성 (VLM): BLIP, BLIP-2, LLaVA (무료 VLM)
#     * ⚠️ 대부분 영어 입력/출력 중심 → 중간에 번역 모델이 반드시 필요
# * 번역 모델:
#     * MarianMT (Helsinki-NLP/opus-mt-ko-en)
#     * M2M100, NLLB (멀티언어 번역, 성능↑)
#     * 역할:
#         * 사용자 한국어 입력 → 영어로 번역 → VLM 전달
#         * VLM 영어 스토리 → 한국어 번역 → 사용자 최종 출력

# 4. 예상 문제점 & 해결 방안
# 문제 1: 실행 시간 길어짐
# * 여러 모델을 연속 실행 시 몇 분씩 소요
# ✅ 해결
# * 필수만 기본 실행, 나머지는 선택형 버튼으로 처리
# * 중간 결과는 st.session_state에 저장해서 재사용
# 문제 2: 중복 실행으로 인한 이미지 손상
# * 업스케일/노이즈 제거 반복 적용 시 artifacts 가능성
# ✅ 해결
# * 기본적으로 버튼 비활성화 (중복 실행 막음)
# * “고급 옵션(실험적)” 체크박스 제공
#     * 체크 시 버튼 활성화
#     * 진행 전 안내: “⚠ 동일 작업 반복은 시간 증가/손상 위험 → 실험적 모드”
# * 반복 횟수 최대 3회 제한
# 문제 3: UX 혼란
# * 지금 보고 있는 이미지가 원본인지, 복원본인지 모호
# ✅ 해결
# * 각 단계 결과에 캡션 + 상태 표시
# * 원본 이미지는 항상 별도 고정 출력
# 문제 4: Streamlit 전체 rerun
# * 버튼 클릭 시 전체 rerun → 결과 유실 위험
# ✅ 해결
# * st.session_state 사용
# * 복원 이미지를 state에 저장 후 재활용
# 문제 5: 스토리 품질 의존성 + 언어 제약
# * 복원 결과가 이상하면 스토리도 이상해짐
# * VLM은 영어 중심 → 한국어 직접 입출력 불안정
# ✅ 해결
# * 스토리 생성 버튼은 사용자 확인 후 실행
# * 원본/복원본 중 선택 가능
# * 번역 모델을 반드시 중간에 둬서 “사용자 ↔ VLM” 간 언어 갭 해소

# 5. 최종 시나리오
# 1. 사용자 사진 업로드 + 흑백/컬러 선택 + 간단 설명(한국어)
# 2. 흑백 → 컬러화 복원 자동 실행 → 결과 표시
# 3. 컬러 → 원본 출력
# 4. 옵션 버튼 활성화 (해상도/노이즈/스토리)
#     * 중복 실행 기본 차단
#     * “고급 옵션” 체크 시 반복 실행 가능 (최대 3회)
# 5. 스토리 생성:
#     * 한국어 입력 설명 → 번역(한→영) → VLM에 전달
#     * VLM이 스토리 생성(영어)
#     * 결과 스토리 → 번역(영→한)
# 6. 최종 출력: 복원 이미지 + 한국어 스토리

In [ ]:
## 모델링

공통
1. BSRGAN(품질복원) + DeOldify(채색)
(필요시 추가 모델 적용)

Track1
2. YOLO 모델(클래스 추출)
YOLO를 이용해 특정 몇몇 클래스를 추출하여 해당 클래스(키워드)를 이용하여 문장을 생성
(alpha) 사용자 텍스트 입력에 대한 처리도 필요
3. 자연어 생성(챗봇 모델 개발)
키워드 기반의 문장을 생성하는 간단한 모델 개발

Track2
(alpha) 사용자 텍스트 입력에 대한 처리도 필요
2. GPT api를 활용하여 이미지를 입력했을 때 자연스러운 문장을 추출하도록 프롬프팅 필요

Track3
2. VLM 모델을 사용(사양에 맞는 모델 사용을 고려) + 사용자의 입력을 어떻게 받는지 모델별 테스트 필요

+위 모델 중 일부 혹은 전부 파인튜닝없이 바로 사용하도록 진행할 것으로 생각중
